# 05.11_CAFE5_node12_Enrich_R

节点功能富集/系统发育可视化分支。

- 当前文件：`analysis/05_genome_analysis/05.11_CAFE5_node12_Enrich_R.ipynb`
- 原始来源：`Codes/05.11_R_CAFE5_node12_Enrich.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`Seurat`, `clusterProfiler`, `data.table`, `dplyr`, `ggplot2`, `ontologyIndex`, `stringr`, `tidyverse`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


r_base

### 0.Set up the environment

In [ ]:
library(data.table)
library(stringr)
library(dplyr)
library(Seurat)
library(ggplot2)
library(clusterProfiler)
library(tidyverse)
library(ontologyIndex)

### 1.Preparing Term to Gene table

In [ ]:
# prepare the term to gene table
eggNOG <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/nodes_eggNOG_processed/node12.emapper.annotations") %>%
    dplyr::select(GOs, `query`) %>%
    dplyr::filter(GOs != "-") %>%
    separate_rows(GOs, sep = ",") %>%
    # mutate(gene = gsub("\\..*", "", `query`)) %>%
    select(GOs, gene = query) %>%
    distinct() %>%
    drop_na()
colnames(eggNOG) <- c("term", "gene")

In [ ]:
head(eggNOG)

### 2.Preparing Term to name table

In [ ]:
# prepare the term to name table
ontology <- get_ontology(file = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/go.obo",
                         propagate_relationships = "is_a",
                         extract_tags = "everything",
                         merge_equivalent_terms = TRUE)
eggNOG_term <- eggNOG %>%
    mutate(name = ontology$name[term]) %>%
    select(c(term, name)) %>%
    distinct() %>%
    drop_na() %>%
    filter(!grepl("obsolete", name))

eggNOG <- eggNOG %>%
    filter(term %in% eggNOG_term$term)

In [ ]:
# node12
write_tsv(x = eggNOG, file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/terms_lib/node12_term2gene_GO.tsv")
write_tsv(x = eggNOG_term, file = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/terms_lib/node12_term2name_GO.tsv")

### 3.Background gene list

In [ ]:
# eggNOG中的基因
background_genes <- eggNOG$gene |> unique()
length(background_genes)

In [ ]:
class(background_genes)
is.vector(background_genes)

### 4.The gene set of interest

In [ ]:
# node12significant.expand.genes.txt
interesting_genes <- read.delim (file = '/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/Nodes_analysis/node12significant.expand.genes.txt', stringsAsFactors = FALSE,header = F)$V1
length(interesting_genes)

### 5.ORA via clusterProfiler

In [ ]:
# node12
term2gene <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/terms_lib/node12_term2gene_GO.tsv")
term2name <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/terms_lib/node12_term2name_GO.tsv")

In [ ]:
# perform ORA
enrichment <- enricher(interesting_genes,
                       TERM2GENE = term2gene,
                       TERM2NAME = term2name,
                       pvalueCutoff = 0.05,
                       universe = background_genes,
                       qvalueCutoff = 0.05)
#save the enrichment result
write.csv(file = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/zz_results/node12_enrichment_GO_results.csv"),                 # EDIT THIS
          x = enrichment@result)

if (any(enrichment@result$p.adjust <= 0.05)){
    p <- dotplot(enrichment,
                 x= "geneRatio", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                 color="p.adjust",
                 orderBy = "x", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                 showCategory=20,
                 font.size=8,
                 label_format = 200 # 标签属于长度
                 ) +
        ggtitle("dotplot for GO ORA")
    
    ggsave(filename = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/zz_results/node12_enrichment_GO_dotplot.pdf"),                # EDIT THIS
           plot =  p,  dpi = 300, width = 8, height = 8)
}

### 6.KEGG

wget https://rest.kegg.jp/link/ko/pathway -O ko_pathway_link.txt
wget https://rest.kegg.jp/list/pathway/ko -O ko_pathway_list.txt

In [ ]:
# === 1. 构建 term2gene / term2name ===
link <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/ko_pathway_link.txt", col_names = FALSE)
list <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/ko_pathway_list.txt", col_names = FALSE)

term2gene_kegg <- link %>%
  filter(grepl("^path:ko", X1)) %>%  # 只保留 path:ko 开头的行
  mutate(
    term = gsub("path:ko", "", X1),   # Pathway ID
    gene = gsub("ko:", "", X2)        # KO ID
  ) %>%
  select(term, gene) %>%
  distinct()

term2name_kegg <- list %>%
  mutate(
    term = gsub("ko", "", X1),        # 保证 term 与 term2gene 一致
    name = X2
  ) %>%
  select(term, name)

In [ ]:
head(term2gene_kegg)

In [ ]:
head(term2name_kegg)

In [ ]:
# === 2. 从 eggNOG 中提取 KEGG 对应 ===
eggNOG_kegg <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/nodes_eggNOG_processed/node12.emapper.annotations") %>%
    dplyr::select(KEGG_ko, `query`) %>%
    dplyr::filter(KEGG_ko != "-") %>%
    separate_rows(KEGG_ko, sep = ",") %>%
    dplyr::mutate(gene = query) %>%
    dplyr::mutate(term = gsub("ko:", "", KEGG_ko)) %>%
    dplyr::select(term, gene) %>%
    distinct() %>%
    drop_na()
    
# === 3. 定义基因集合 ===
interesting_set_kegg <- eggNOG_kegg %>%
    dplyr::filter(gene %in% interesting_genes) %>%
    unlist() %>%
    as.vector()
# create a list of kegg ortholog that includes all kegg orthologs which form my background
background_kegg <- eggNOG_kegg %>%
    dplyr::filter(gene %in% background_genes) %>%
    unlist() %>%
    as.vector()

# enrichment_kegg <- enrichKEGG(interesting_set_kegg,
#            organism = "ko",
#            keyType = "kegg",
#            pvalueCutoff = 0.05,
#            pAdjustMethod = "BH",
#            universe = background_kegg,
#            minGSSize = 10,
#            maxGSSize = 500,
#            qvalueCutoff = 0.05,
#            use_internal_data = FALSE)

# === 4. 运行离线 KEGG 富集 ===
enrichment_kegg <- enricher(
  gene = interesting_set_kegg,        # 感兴趣基因（KO ID 或基因名）
  TERM2GENE = term2gene_kegg,      # pathway–gene 对照表
  TERM2NAME = term2name_kegg,      # pathway 名称表
  universe = background_kegg,     # 背景基因
  pAdjustMethod = "BH",
  pvalueCutoff = 0.05,
  qvalueCutoff = 0.05
)
#save the enrichment result
write.csv(file = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/zz_results/node12_enrichment_KEGG_results.csv"),                 # EDIT THIS
          x = enrichment_kegg@result)

if (any(enrichment_kegg@result$p.adjust <= 0.05)){
    p <- dotplot(enrichment_kegg,
                 x= "geneRatio", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                 color="p.adjust",
                 orderBy = "x", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                 showCategory=20,
                 font.size=8) +
        ggtitle("dotplot for KEGG ORA")
    
    ggsave(filename = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/target_nodes_analysis/zz_results/node12_enrichment_KEGG_dotplot.pdf"),                # EDIT THIS
           plot =  p,  dpi = 300, width = 8, height = 8)
}
